In [16]:
import pandas as pd
import random
import os
from sklearn.model_selection import train_test_split  # <-- The new powerhouse import

In [17]:
# =====================================================================
# 1. BASELINE DICTIONARIES (Anchored to Reality)
# =====================================================================
proximity_map = {
    'Amaya': 1.8, 'Bagtas': 6.9, 'Biga': 4.8, 'Biwas': 2.1, 'Bucal': 1.9, 
    'Bunga': 1.7, 'Calibuyo': 6.8, 'Capipisa': 9.0, 'Daang Amaya': 0.8,
    'Halayhay': 6.1, 'Julugan': 3.0, 'Lambingan': 10.4, 'Mulawin': 1.5, 
    'Paradahan': 8.9, 'Poblacion': 0.6, 'Punta': 4.4, 'Sahud Ulan': 5.9, 
    'Sanja Mayor': 2.5, 'Santol': 3.0, 'Tanauan': 17.6, 'Tres Cruses': 9.6
}

flood_baseline_map = {
    'Capipisa': 4.5, 'Julugan': 4.2, 'Halayhay': 3.8, 'Poblacion': 3.5, 'Amaya': 3.2, 
    'Calibuyo': 2.8, 'Sahud Ulan': 2.5, 'Sanja Mayor': 2.2, 'Daang Amaya': 2.0, 
    'Biwas': 1.8, 'Bagtas': 1.5, 'Mulawin': 1.5, 'Punta': 1.2, 'Biga': 1.0, 
    'Lambingan': 0.8, 'Bucal': 0.5, 'Santol': 0.5, 'Bunga': 0.4, 'Tanauan': 0.3, 
    'Paradahan': 0.2, 'Tres Cruses': 0.1
}

coastal = ['Julugan', 'Capipisa', 'Halayhay', 'Amaya', 'Poblacion']
highway = ['Daang Amaya', 'Biwas', 'Biga', 'Mulawin', 'Sanja Mayor', 'Punta']

In [18]:
# =====================================================================
# 2. GENERATION PARAMETERS
# =====================================================================
START_YEAR = 2021
END_YEAR = 2025
RECORDS_PER_YEAR_PER_BRGY = 5  
ANNUAL_INFLATION_RATE = 0.06     
BASE_PRICE_START = 2500000       
REGIONAL_BASE_AQI = 72.0         

dataset = []

print("⏳ Generating 2020-2025 Digital Twin historical dataset...")

for year in range(START_YEAR, END_YEAR + 1):
    years_passed = year - START_YEAR
    
    yearly_aqi_creep = years_passed * 0.5 
    yearly_flood_creep = years_passed * 0.02
    
    current_base_price = BASE_PRICE_START * ((1 + ANNUAL_INFLATION_RATE) ** years_passed)
    
    for brgy, prox in proximity_map.items():
        base_aqi = REGIONAL_BASE_AQI + yearly_aqi_creep
        if brgy in coastal: base_aqi -= 6
        if brgy in highway: base_aqi += 8
            
        base_flood = flood_baseline_map[brgy] + yearly_flood_creep
            
        for _ in range(RECORDS_PER_YEAR_PER_BRGY):
            house_aqi = max(10, round(random.gauss(base_aqi, 3), 1))
            house_flood = max(0.0, round(random.gauss(base_flood, 0.4), 2))
            
            prox_penalty = (prox / 10.0) * 0.40 
            env_penalty = ((house_aqi / 150.0) * 0.10) + ((house_flood / 5.0) * 0.15)
            
            total_penalty_multiplier = 1.0 - prox_penalty - env_penalty
            market_noise = random.uniform(0.90, 1.10) 
            
            final_price = current_base_price * total_penalty_multiplier * market_noise
            
            record_id = f"{brgy[:3].upper()}-{year}-{random.randint(1000, 9999)}"
            
            dataset.append({
                'Record_ID': record_id,
                'Year': year,
                'Barangay': brgy,
                'Proximity_km': prox,
                'AQI': house_aqi,
                'Flood_Risk': house_flood,
                'Price_PHP': round(final_price)
            })

⏳ Generating 2020-2025 Digital Twin historical dataset...


In [19]:
# =====================================================================
# 3. DIRECTORY SETUP & STRICT 80/20 STRATIFIED SPLIT
# =====================================================================
print("📁 Setting up directories and splitting data...")

os.makedirs('dataset/train', exist_ok=True)
os.makedirs('dataset/test', exist_ok=True)

full_df = pd.DataFrame(dataset)

# The new scikit-learn split. 
# 'stratify=full_df['Barangay']' guarantees every barangay gets an exact 80/20 cut!
train_df, test_df = train_test_split(
    full_df, 
    test_size=0.20, 
    stratify=full_df['Barangay'], 
    random_state=42
)

📁 Setting up directories and splitting data...


In [20]:
# =====================================================================
# 4. ENTITY SEPARATION & EXPORT
# =====================================================================
def save_split_data(df, folder_path, suffix):
    aqi_df = df[['Record_ID', 'Year', 'Barangay', 'AQI']]
    aqi_df.to_csv(f"{folder_path}/air_quality_{suffix}.csv", index=False)
    
    flood_df = df[['Record_ID', 'Year', 'Barangay', 'Flood_Risk']]
    flood_df.to_csv(f"{folder_path}/flood_risk_{suffix}.csv", index=False)
    
    housing_df = df[['Record_ID', 'Year', 'Barangay', 'Proximity_km', 'Price_PHP']]
    housing_df.to_csv(f"{folder_path}/housing_data_{suffix}.csv", index=False)

save_split_data(train_df, 'dataset/train', 'train')
save_split_data(test_df, 'dataset/test', 'test')

print("✅ Success! Your Digital Twin dataset is completely generated.")
print(f"Total Records: {len(full_df)} (Train: {len(train_df)}, Test: {len(test_df)})")
print("Check your 'dataset/' folder for the separated CSVs.")

✅ Success! Your Digital Twin dataset is completely generated.
Total Records: 525 (Train: 420, Test: 105)
Check your 'dataset/' folder for the separated CSVs.
